# Model Inference and Interpretability: From Linear Models to Post-Hoc Explainability
Accompanies **Unit 9.1: Model Inference and Interpretability**.

In production and scientific applications, predicting accurately is only half the battle. Stakeholders require:
- **Statistical Inference**: What are the confidence intervals on parameters? What is the effect size?
- **Global Explainability**: Which features drive the model's decisions, and what are their non-linear response profiles?
- **Local Recourse (Instance Explainability)**: Why was a specific applicant rejected, and what exact factors caused that prediction?

In this notebook, we cover:
1. **Linear Model Inference**: OLS summary tables, standard errors, $t$-tests, $p$-values, confidence intervals, Cohen's $f^2$ effect size, and logistic odds ratios.
2. **Non-Linear Global Explainability**: Permutation Feature Importance vs. Impurity Importance (MDI), Partial Dependence Plots (PDP), and Individual Conditional Expectation (ICE).
3. **Local Explainability**: Local Interpretable Model-agnostic Explanations (LIME) and SHapley Additive exPlanations (SHAP).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

data_path = os.path.abspath('../../data/loan_data_train.csv')
df = pd.read_csv(data_path)

# Prepare clean subset
df_clean = df[['Amount.Requested', 'Monthly.Income', 'Open.CREDIT.Lines', 'Revolving.CREDIT.Balance', 'Interest.Rate']].copy()
df_clean['Interest.Rate'] = pd.to_numeric(df_clean['Interest.Rate'].astype(str).str.replace('%', '', regex=False), errors='coerce')
df_clean = df_clean.dropna()

feature_names = ['Amount.Requested', 'Monthly.Income', 'Open.CREDIT.Lines', 'Revolving.CREDIT.Balance']
X = df_clean[feature_names].values
y = df_clean['Interest.Rate'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## 1. Linear Model Inference: Statsmodels OLS and Cohen's $f^2$
In machine learning, we predict; in statistics, we infer.
Under Gauss-Markov assumptions:
$$\hat{\boldsymbol{\beta}} \sim \mathcal{N}\left(\boldsymbol{\beta}, \sigma^2 (\mathbf{X}^T \mathbf{X})^{-1}\right)$$
We inspect the OLS summary table to examine standard errors, $t$-statistics, $p$-values, and $95\%$ confidence intervals.

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols_res = sm.OLS(y_train, X_train_sm).fit()

print(ols_res.summary(xname=['const'] + feature_names))

### Computing Cohen's $f^2$ Effect Size
In large samples, $p$-values vanish toward zero even for trivial effects.
Cohen's $f^2$ evaluates the practical, physical contribution of each feature to explained variance:
$$f^2 = \frac{R_{\text{full}}^2 - R_{\text{reduced}}^2}{1 - R_{\text{full}}^2}$$
Benchmarks: $0.02$ (small), $0.15$ (medium), $0.35$ (large).

In [ ]:
r2_full = ols_res.rsquared

print("--- Cohen's f^2 Effect Sizes ---")
for i, feat in enumerate(feature_names):
    # Fit reduced model excluding feature i
    cols = [c for c in range(1, X_train_sm.shape[1]) if c != (i + 1)]
    reduced_model = sm.OLS(y_train, X_train_sm[:, [0] + cols]).fit()
    r2_reduced = reduced_model.rsquared
    f2 = (r2_full - r2_reduced) / (1.0 - r2_full)
    category = "Large" if f2 >= 0.35 else ("Medium" if f2 >= 0.15 else ("Small" if f2 >= 0.02 else "Negligible"))
    print(f"  {feat:25s}: f2 = {f2:.4f} ({category})")

## 2. Logistic Regression Inference: Odds Ratios and Marginal Effects
For binary classification, exponentiating coefficients yields Odds Ratios:
$$\text{OR}_j = \exp(\beta_j)$$
Average Marginal Effects (AME) translate non-linear log-odds shifts into percentage point changes:
$$\text{AME}_j = \frac{1}{N}\sum_{i=1}^N p_i(1-p_i)\beta_j$$

In [ ]:
bd_path = os.path.abspath('../../data/bd_train.csv')
df_bd = pd.read_csv(bd_path)

logit_features = ['age', 'balance', 'duration', 'campaign']
X_b = df_bd[logit_features].fillna(df_bd[logit_features].median())
y_b = (df_bd['y'] == 'yes').astype(int)

# Fit Logit model
X_b_sm = sm.add_constant(StandardScaler().fit_transform(X_b))
logit_model = sm.Logit(y_b, X_b_sm).fit(disp=False)

# Odds Ratios and 95% Confidence Intervals
params = logit_model.params[1:]
conf = logit_model.conf_int()[1:]
odds_df = pd.DataFrame({
    'Feature': logit_features,
    'Odds Ratio': np.exp(params),
    '2.5% CI': np.exp(conf[0]),
    '97.5% CI': np.exp(conf[1])
})
print("--- Logistic Odds Ratios ---")
print(odds_df.to_string(index=False))

# Average Marginal Effects
margeff = logit_model.get_margeff()
print("\n--- Average Marginal Effects (AME) ---")
print(margeff.summary())

## 3. Non-Linear Models: Permutation Importance vs. Impurity (MDI)
Fit a Random Forest to compare tree-based MDI against model-agnostic Permutation Importance on holdout test data.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# MDI Importance
mdi_imp = rf.feature_importances_

# Permutation Importance on holdout test data
perm_res = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_imp = perm_res.importances_mean

comp_df = pd.DataFrame({
    'Feature': feature_names,
    'MDI (In-Sample)': mdi_imp,
    'Permutation (Test Holdout Drop)': perm_imp
}).sort_values(by='Permutation (Test Holdout Drop)', ascending=False)

print("--- Feature Importance Comparison ---")
print(comp_df.to_string(index=False))

## 4. Partial Dependence Plots (PDP) and Individual Conditional Expectation (ICE)
- **PDP**: Displays the average marginal effect of a feature on predictions: $\bar{f}(x_S) = \frac{1}{N}\sum f(x_S, \mathbf{x}_{i, C})$.
- **ICE**: Displays individual curves for every single observation, exposing heterogeneous interactions that flat PDP averages mask.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
PartialDependenceDisplay.from_estimator(
    rf, X_train, features=[0, 1], feature_names=feature_names,
    kind='both', subsample=50, ax=ax, line_kw={'color': 'royalblue'}
)
plt.suptitle('Partial Dependence (Thick Line) & ICE Curves (Thin Lines)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Local Interpretability: LIME (Local Interpretable Model-Agnostic Explanations)
LIME explains an individual prediction by generating local perturbations, evaluating black-box outputs, and fitting a local distance-weighted linear surrogate.

In [ ]:
from sklearn.linear_model import Ridge

# Select a single test instance to explain
instance_idx = 5
x_target = X_test[instance_idx]
pred_target = rf.predict(x_target.reshape(1, -1))[0]

# Generate 500 perturbations around x_target
np.random.seed(42)
pert_samples = np.random.normal(loc=x_target, scale=np.std(X_train, axis=0) * 0.25, size=(500, len(x_target)))
pert_preds = rf.predict(pert_samples)

# Proximity weights using exponential kernel
dists = np.linalg.norm(pert_samples - x_target, axis=1)
weights = np.exp(-(dists ** 2) / (np.mean(dists) ** 2))

# Fit local linear model
local_surrogate = Ridge(alpha=1.0)
local_surrogate.fit(pert_samples, pert_preds, sample_weight=weights)

print(f"Target Instance Actual Interest Rate:    {y_test[instance_idx]:.2f}%")
print(f"Random Forest Predicted Interest Rate:   {pred_target:.2f}%")
print(f"Local LIME Surrogate Intercept (w0):    {local_surrogate.intercept_:.2f}%")
print("\nLocal LIME Explanatory Weights:")
for feat, w in zip(feature_names, local_surrogate.coef_):
    print(f"  {feat:25s}: {w:+.4f} (% shift per unit change)")